# MultiGEC (GEC) pilot analysis

Two questions, from the pilot runs in `Experiment_results_publication/Archived_results/GEC`:

1. **Runtime** per model x reasoning arm, and what the full 504-example run would cost.
2. **Wrong-text audit** — is `wrong_text_rate` 0 under constrained decoding, and is any
   non-zero value explained by the model hitting `max_new_tokens` rather than by a
   verbatim-copy violation?

> **Caveats.** Pilot runs: 10 examples, one seed. MultiGEC is a single 504-example file
> with no subsets, so the extrapolation is one rate x one corpus size — simpler than the
> UNER/WMT notebooks, which had to handle per-subset sizes.

Budget for the full runs (`PUBLICATION_PLAN.md`): **24 h hard ceiling, 16–18 h target**,
**one seed**.

In [1]:
import sys, json, glob, re
from pathlib import Path
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "utils").is_dir())
sys.path.insert(0, str(ROOT / "src"))
RES = ROOT / "Experiment_results_publication" / "Archived_results" / "GEC"
assert RES.is_dir(), RES

pd.set_option("display.width", 250)
pd.set_option("display.max_rows", 200)

from utils.span_datasets import load_multigec
FULL_N = len(load_multigec())
print(f"MultiGEC full size: {FULL_N} examples")

frames = []
for f in sorted(glob.glob(str(RES / "Csv/*.csv"))):
    df = pd.read_csv(f)
    if df.empty:
        continue
    df["file"] = Path(f).name
    frames.append(df)

runs = pd.concat(frames, ignore_index=True)
runs["model_short"] = runs["model"].str.split("/").str[-1]
runs["effort"] = runs["reasoning_effort"].fillna("n|a")
runs["arm"] = runs.apply(
    lambda r: f"{'ON' if r['reasoning_enabled'] else 'OFF'}"
              + (f"/{r['effort']}" if r["effort"] != "n|a" else ""), axis=1)

print(f"{len(frames)} CSV files -> {len(runs)} rows "
      f"({runs['model_short'].nunique()} models, seeds={sorted(runs['n_iters'].unique())})")

MultiGEC full size: 504 examples
10 CSV files -> 20 rows (6 models, seeds=[np.int64(1)])


## 1. Runtime and projection to the full 504 examples

`elapsed_minute_avg` is wall time for one seed, one `(eval_mode, processor_class)`.
A submitted job runs both `unconstrained` and `constrained`, so a job's cost is the two
added together — `job_min`.

In [2]:
wide = (runs
    .pivot_table(index=["model_short", "arm", "max_examples", "max_new_tokens"],
                 columns="eval_mode", values="elapsed_minute_avg", aggfunc="mean")
    .reset_index())
for col in ("constrained", "unconstrained"):
    if col not in wide.columns:
        wide[col] = float("nan")

wide["job_min"] = wide["constrained"].fillna(0) + wide["unconstrained"].fillna(0)
wide = wide.rename(columns={"constrained": "cons_min", "unconstrained": "uncons_min",
                            "max_examples": "n_ex"})
wide["sec_per_ex_job"] = wide["job_min"] * 60 / wide["n_ex"]
wide["h_full_1seed"] = wide["sec_per_ex_job"] * FULL_N / 3600

out = wide.sort_values("h_full_1seed", ascending=False)
print(f"Projected hours for the FULL {FULL_N} examples, ONE seed, one job "
      f"(unconstrained + constrained):")
print(out[["model_short", "arm", "n_ex", "cons_min", "uncons_min",
           "sec_per_ex_job", "h_full_1seed"]].round(2).to_string(index=False))

print()
print(f"  over 24h (hard ceiling): {(out.h_full_1seed > 24).sum()} / {len(out)}")
print(f"  over 18h (target)      : {(out.h_full_1seed > 18).sum()} / {len(out)}")

Projected hours for the FULL 504 examples, ONE seed, one job (unconstrained + constrained):
   model_short       arm  n_ex  cons_min  uncons_min  sec_per_ex_job  h_full_1seed
   Qwen3.8-27B    ON/low    10     34.40       33.26          405.91         56.83
  gpt-oss-120b ON/medium    10     24.66       24.65          295.81         41.41
   gpt-oss-20b ON/medium    10     23.71       18.66          254.21         35.59
gemma-4-E2B-it        ON    10     10.94       10.58          129.15         18.08
  gpt-oss-120b    ON/low    10      7.71        7.41           90.71         12.70
gemma-4-31B-it       OFF    10      7.35        7.29           87.86         12.30
   Qwen3.8-27B       OFF    10      5.05        5.38           62.60          8.76
   gpt-oss-20b    ON/low    10      3.60        4.29           47.37          6.63
gemma-4-E2B-it       OFF    10      1.97        1.86           22.99          3.22
      Qwen3-8B       OFF    10      1.53        1.49           18.11          

## 2. Wrong-text audit

The claim under test: **under constrained decoding the wrong-text rate is 0, conditional
on reasoning terminating.** Any non-zero value must be a *truncation* failure — the model
spent its whole `max_new_tokens` budget reasoning and never emitted an answer — never a
verbatim-copy violation.

`max_new_tokens` is not in the JSONL rows, so it comes from the CSVs; the next cell
asserts it is constant per model rather than assuming it.

In [3]:
budget = runs.groupby("model_short")["max_new_tokens"].agg(["nunique", "max"])
assert (budget["nunique"] == 1).all(), f"max_new_tokens varies within a model:\n{budget}"
BUDGET = budget["max"].to_dict()
print("token budget per model:", BUDGET)

PRED = re.compile(r"^(?P<ds>.+?)_(?P<model>[^_]+(?:-[^_]+)*)_think_(?P<think>True|False)"
                  r"_(?P<samp>sampling|greedy)_(?P<mode>constrained|unconstrained)"
                  r"_(?P<cfg>think\d.*?)_(?P<proc>n\|a|token_aware)$")

rows = []
for f in sorted(glob.glob(str(RES / "Predictions/*.jsonl"))):
    m = PRED.match(Path(f).stem)
    if not m:
        print("UNPARSED filename (skipped):", Path(f).name)
        continue
    g = m.groupdict()
    for line in open(f, encoding="utf-8"):
        if not line.strip():
            continue
        r = json.loads(line)
        rows.append(dict(
            model=g["model"], mode=g["mode"], cfg=g["cfg"],
            budget=BUDGET.get(g["model"]),
            wrong=r["wrong_text"], ntok=r["num_output_tokens"],
            rtok=r.get("num_reasoning_tokens"), atok=r.get("num_answer_tokens"),
            found_end=r.get("found_reasoning_end"), skipped=r.get("reasoning_skipped"),
            gold_empty=r.get("gold_empty_spans"), pred_empty=r.get("pred_empty_spans"),
        ))

pred = pd.DataFrame(rows)
pred["hit_cap"] = pred["ntok"] >= pred["budget"]
print(f"\n{len(pred)} prediction rows; budget resolved for {pred['budget'].notna().sum()}")
print(pred.groupby("mode").agg(rows=("wrong", "size"), wrong=("wrong", "sum")).to_string())

token budget per model: {'Qwen3-8B': 18000, 'Qwen3.8-27B': 18000, 'gemma-4-31B-it': 18000, 'gemma-4-E2B-it': 18000, 'gpt-oss-120b': 16000, 'gpt-oss-20b': 16000}

200 prediction rows; budget resolved for 200
               rows  wrong
mode                      
constrained     100      1
unconstrained   100     36


In [4]:
cons = pred[pred["mode"] == "constrained"]
bad = cons[cons["wrong"] == 1]

print(f"CONSTRAINED rows: {len(cons)}   wrong_text: {len(bad)}   "
      f"rate: {100*len(bad)/max(len(cons),1):.2f}%")
print()
if len(bad):
    print("Every constrained wrong_text row, with its explanation:")
    print(bad[["model", "cfg", "ntok", "budget", "hit_cap",
               "rtok", "atok", "found_end", "skipped"]].to_string(index=False))
    print()
    unexplained = bad[~bad["hit_cap"].fillna(False)]
    print(f"  hit the token cap   : {int(bad['hit_cap'].sum())}")
    print(f"  did NOT hit the cap : {len(unexplained)}   <-- must be 0")
    if len(unexplained):
        print("\n  !! UNEXPLAINED verbatim-copy violations:")
        print(unexplained.to_string(index=False))
else:
    print("No constrained wrong_text rows at all.")

print()
print("Constrained wrong_text by whether reasoning terminated:")
c = cons.copy()
c["terminated"] = c["found_end"].fillna(True) | c["skipped"].fillna(False)
print(c.groupby("terminated").agg(rows=("wrong", "size"), wrong=("wrong", "sum")).to_string())
print()
print("^ The claim: wrong_text is 0 wherever reasoning terminated.")

CONSTRAINED rows: 100   wrong_text: 1   rate: 1.00%

Every constrained wrong_text row, with its explanation:
      model                  cfg  ntok  budget  hit_cap  rtok  atok  found_end  skipped
gpt-oss-20b think1_effort_medium 16000   16000     True 16000     0      False    False

  hit the token cap   : 1
  did NOT hit the cap : 0   <-- must be 0

Constrained wrong_text by whether reasoning terminated:
            rows  wrong
terminated             
False         41      1
True          59      0

^ The claim: wrong_text is 0 wherever reasoning terminated.


### GEC-specific: did the models use the empty-tag convention?

`M` is a zero-length insertion point, so the model has to emit `<SPAN><LABEL>M</LABEL></SPAN>`
with nothing inside. If `pred_empty` is 0 everywhere the models never used the convention,
which is a *prompting* failure — indistinguishable from a constraint failure in F1 alone,
which is why the field is logged separately.

In [5]:
emp = (pred.groupby(["mode", "model"])
       .agg(rows=("wrong", "size"),
            gold_empty=("gold_empty", "sum"),
            pred_empty=("pred_empty", "sum"))
       .reset_index())
print(emp.to_string(index=False))

         mode          model  rows  gold_empty  pred_empty
  constrained       Qwen3-8B    10          41           0
  constrained    Qwen3.8-27B    20          82           0
  constrained gemma-4-31B-it    10          41           6
  constrained gemma-4-E2B-it    20          82           0
  constrained   gpt-oss-120b    20          82          19
  constrained    gpt-oss-20b    20          82           4
unconstrained       Qwen3-8B    10          41           5
unconstrained    Qwen3.8-27B    20          82          28
unconstrained gemma-4-31B-it    10          41           8
unconstrained gemma-4-E2B-it    20          82           0
unconstrained   gpt-oss-120b    20          82          21
unconstrained    gpt-oss-20b    20          82          10


## Summary

Fill in after running:

- **Runtime** — table 1, `h_full_1seed` per model+arm against the 18 h target / 24 h ceiling.
- **Wrong text** — constrained rate, and whether every non-zero case hit `max_new_tokens`.
- **Empty spans** — whether the `M` convention was used at all.